In [ ]:
!pip install python-dotenv huggingface-hub llama-index transformers sentence-transformers llama-index-llms-huggingface llama-index-embeddings-huggingface pdfplumber llama-index-llms-openrouter llama-index-retrievers-bm25 tabula-py  jpype1 pystemmer

In [2]:
!pip install --upgrade pip
!pip install -q pytesseract pdf2image pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 20.6 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.3
    Uninstalling pip-25.3:
      Successfully uninstalled pip-25.3


## 1. Setup & Configuration
In this section, we set up the necessary dependencies, including third-party libraries and API keys.
We also configure the language models (LLMs) and embedding models used throughout the tutorial.

Key Steps:
- Import required Python libraries.
- Load API keys securely from environment variables.
- Initialize the OpenRouter LLMs for both querying and evaluation.
- Set up the HuggingFace embedding model for text representation.
- Apply `nest_asyncio` to handle event loop issues in Jupyter environments.


In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3" # Suppresses TensorFlow warnings
from dotenv import load_dotenv
import Stemmer
import nest_asyncio
import tabula
import pandas as pd
from dotenv import load_dotenv
from llama_index.core import Document
from llama_index.core import (SimpleDirectoryReader, VectorStoreIndex, Settings)
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.openrouter import OpenRouter
import asyncio


load_dotenv()  # reads OPENROUTER_API_KEY from a local .env file (never commit it)
api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    raise ValueError("OPENROUTER_API_KEY not found. Copy .env.example to .env and add your key.")


# Initialize OpenRouter LLM
#llm = OpenRouter(api_key=api_key, model="mistralai/mistral-7b-instruct", max_tokens=512, context_window=4096)
llm = OpenRouter(api_key=api_key, model="meta-llama/Llama-3.1-8B-Instruct", max_tokens=512, context_window=4096)
Judge_llm = OpenRouter(api_key=api_key, model="qwen/qwen-turbo", max_tokens=512, context_window=4096)
Settings.llm = llm

# Apply nest_asyncio to fix event loop issues in Jupyter
nest_asyncio.apply()

# Set up embedding model
embed_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embed_model = HuggingFaceEmbedding(model_name=embed_model_name)
Settings.embed_model = embed_model

## 2. Document Loading & Preprocessing
Here, we load the syllabus document from a PDF file and process it for retrieval.

Key Steps:
- Read the syllabus PDF and extract its textual content.
- Extract tabular data from the PDF using `tabula`.
- Convert the extracted table data into text format.
- Combine the extracted text and tables into a unified document.
- Define an ingestion pipeline to preprocess text by splitting it into manageable chunks and applying embeddings.


In [2]:
import warnings
import logging
warnings.filterwarnings("ignore")
logging.getLogger().setLevel(logging.ERROR)

for name in [
    "llama_index",
    "httpx",
    "httpcore",
    "openai",
    "urllib3",
    "tabula",
    "py4j",
]:
    logging.getLogger(name).setLevel(logging.ERROR)

In [3]:
import re
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter, TokenTextSplitter
from llama_index.core.ingestion import IngestionPipeline
from pdf2image import convert_from_path, pdfinfo_from_path
import pytesseract

# Load Course Syllabus Document
pdf_path = "./WebMD.pdf"
documents = SimpleDirectoryReader(input_files=[pdf_path]).load_data()

# Convert table data into an additional Document
tables = tabula.read_pdf(pdf_path, pages="all", force_subprocess=True)
table_docs = [df.to_markdown(index=False) for df in tables]
all_tables_text = "\n\n".join(table_docs)
document_from_tables = Document(text=all_tables_text,metadata={"source": "tabula_tables", "file": pdf_path})

documents = documents + [document_from_tables]

# PDF TEXT CLEANING STEP 
def clean_pdf_text(t: str) -> str:
    t = re.sub(r"/gid\d+", "", t) 
    t = re.sub(r"(\w)-\s*\n(\w)", r"\1\2", t) 
    t = re.sub(r"\n{3,}", "\n\n", t) 
    t = re.sub(r"[ \t]{2,}", " ", t)
    return t.strip()

documents = [
    Document(text=clean_pdf_text(d.text), metadata=d.metadata)
    for d in documents
]

def ocr_pdf_pages(pdf_path, pages=None, dpi=300):
    total_pages = pdfinfo_from_path(pdf_path)["Pages"]
    ocr_texts = []

    # If pages not provided, OCR all
    if pages is None:
        pages = list(range(1, total_pages + 1))

    for p in pages:
        if p < 1 or p > total_pages:
            print(f"Skipping page {p} (out of range). PDF has {total_pages} pages.")
            continue

        imgs = convert_from_path(pdf_path, dpi=dpi, first_page=p, last_page=p)

        if not imgs: 
            print(f"No image rendered for page {p}. Skipping.")
            continue

        text = pytesseract.image_to_string(imgs[0])
        if text.strip():
            ocr_texts.append(f"\n\n--- OCR PAGE {p} ---\n{text}")

    return "\n".join(ocr_texts).strip()

ocr_text = ocr_pdf_pages(pdf_path, pages=[1, 2, 8, 9]) 
print(ocr_text[:500])

# pipeline with transformations
pipeline = IngestionPipeline(
    transformations=[
        TokenTextSplitter(
            separator="\n\n",
            chunk_size=512,
            chunk_overlap=100
        ),
        embed_model
    ]
)

if ocr_text:
    ocr_doc = Document(
        text=ocr_text,
        metadata={"source": "ocr", "file": pdf_path}
    )
    documents = documents + [ocr_doc]

nodes = pipeline.run(documents=documents)

--- OCR PAGE 1 ---
EAL STO

A look inside
your head
and more
page 9

How | help others
help me
page 14

a

Prioritize ( festyle

chan
page



--- OCR PAGE 2 ---
WebMD | CHRONIC MIGRAINE CONTENTS

INSIDE THE ISSUE

NEWS FIRST PERSON

The latest research Being my own advocate

HEALTH SMARTS VIEWPOINT

A look inside your head Here’s how | help others help me
and more through chronic migraine

_ TREATMENT SMARTS LIVING HEALTHY

Prioritize lifestyle changes

WebMD Real Stories is not responsible for 


In [4]:
print("Total nodes:", len(nodes))

ocr_found = False
for n in nodes:
    if "--- OCR PAGE" in n.get_content():
        ocr_found = True
        break

print("OCR text indexed?", ocr_found)

Total nodes: 21
OCR text indexed? True


## 3. Indexing & Retrieval
Once the document is processed, we create multiple retrievers for efficient information retrieval.

Key Steps:
- Build a `VectorStoreIndex` from the preprocessed document chunks.
- Implement different retrieval methods:
  - **Base Retriever:** Retrieves the most relevant document chunk.
  - **AutoMerging Retriever:** Aggregates multiple related chunks before returning results.
  - **BM25 Retriever:** Uses term frequency-based ranking (a traditional information retrieval method).
  - **Hybrid Fusion Retriever:** (Task 4)
- Define retrieval hyperparameters, such as similarity threshold and number of top results to return.


In [5]:
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.retrievers import AutoMergingRetriever

# Vector Index and Query Engine
index = VectorStoreIndex(nodes)
query_engine = index.as_query_engine()

base_retriever = index.as_retriever(similarity_top_k=8)

auto_base_retriever = index.as_retriever(similarity_top_k=3)
auto_merging_retriever = AutoMergingRetriever(auto_base_retriever, index.storage_context)

bm25_retriever = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=2, stemmer=Stemmer.Stemmer("english"), language="english")

# Create Hybrid Fusion Retriever (Task 4)
from llama_index.core.retrievers import QueryFusionRetriever

vector_retriever_fusion = index.as_retriever(similarity_top_k=8)
bm25_retriever_fusion = BM25Retriever.from_defaults(
    nodes=nodes,
    similarity_top_k=8,
    stemmer=Stemmer.Stemmer("english"),
    language="english"
)

fusion_retriever = QueryFusionRetriever(
    retrievers=[vector_retriever_fusion, bm25_retriever_fusion],
    retriever_weights=[0.6, 0.4],
    similarity_top_k=5,   
    num_queries=1        
)

## 4. Query Engines & Evaluation Setup
In this section, we configure the query engines and set up evaluation metrics to assess retriever performance.

Key Steps:
- Instantiate query engines for each retrieval method to allow direct querying.
- Define evaluation models to measure:
  - **Faithfulness:** Whether the retrieved information is accurate and grounded in the original document.
  - **Relevancy:** Whether the retrieved information is relevant to the query.
- Configure different retriever evaluators to assess retrieval effectiveness using metrics such as:
  - Mean Reciprocal Rank (MRR)
  - Hit Rate
  - Precision
  - Recall


In [6]:
# query engines
from llama_index.core.query_engine import RetrieverQueryEngine

base_query_engine = RetrieverQueryEngine.from_args(base_retriever)
auto_query_engine = RetrieverQueryEngine.from_args(auto_merging_retriever)
bm25_query_engine = RetrieverQueryEngine.from_args(bm25_retriever)

from llama_index.core.evaluation import FaithfulnessEvaluator, RelevancyEvaluator, RetrieverEvaluator

faithfulness_evaluator = FaithfulnessEvaluator(llm=Judge_llm)
relevancy_evaluator = RelevancyEvaluator(llm=Judge_llm)

base_retriever_evaluator = RetrieverEvaluator.from_metric_names(["mrr", "hit_rate", "precision", "recall"], retriever=base_retriever)
auto_retriever_evaluator = RetrieverEvaluator.from_metric_names(["mrr", "hit_rate", "precision", "recall"], retriever=auto_merging_retriever)
bm25_retriever_evaluator = RetrieverEvaluator.from_metric_names(["mrr", "hit_rate", "precision", "recall"], retriever=bm25_retriever)

In [9]:
print("===== TASK 3: DEMONSTRATION =====")

test_query = "What section headings and page references are listed on the cover page under 'A look inside your head'?"

response = base_query_engine.query(test_query)

print("\nAnswer:\n")
print(response.response)

print("\nSource Metadata:\n")
for s in response.source_nodes:
    print(s.metadata)

===== TASK 3: DEMONSTRATION =====

Answer:

A look inside your head and more page 9 How | help others help me page 14

Source Metadata:

{'page_label': '1', 'file_name': 'WebMD.pdf', 'file_path': 'WebMD.pdf', 'file_type': 'application/pdf', 'file_size': 10299888, 'creation_date': '2026-02-25', 'last_modified_date': '2026-02-25'}
{'source': 'ocr', 'file': './WebMD.pdf'}
{'page_label': '2', 'file_name': 'WebMD.pdf', 'file_path': 'WebMD.pdf', 'file_type': 'application/pdf', 'file_size': 10299888, 'creation_date': '2026-02-25', 'last_modified_date': '2026-02-25'}
{'page_label': '5', 'file_name': 'WebMD.pdf', 'file_path': 'WebMD.pdf', 'file_type': 'application/pdf', 'file_size': 10299888, 'creation_date': '2026-02-25', 'last_modified_date': '2026-02-25'}
{'page_label': '3', 'file_name': 'WebMD.pdf', 'file_path': 'WebMD.pdf', 'file_type': 'application/pdf', 'file_size': 10299888, 'creation_date': '2026-02-25', 'last_modified_date': '2026-02-25'}
{'page_label': '9', 'file_name': 'WebMD.pdf', 